In [ ]:
"""
This script is a practical example of how to combine two time-series datasets
from different sources. It shows how to standardize date/time fields, align
common columns, stack the datasets safely, keep track of the source of each row,
and check that no data is accidentally lost during the process.
"""

'\nThis script is a practical example of how to combine two time-series datasets\nfrom different sources. It shows how to standardize date/time fields, align\ncommon columns, stack the datasets safely, keep track of the source of each row,\nand check that no data is accidentally lost during the process.\n'

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 1. LOAD DATA
# =========================================================
df1 = pd.read_excel("data for water temp combining/water_quality_with_met_Mar17.xlsx")   # 2022+ dataset
df2 = pd.read_csv("data for water temp combining/fixed_dataset_after_cleaning_gaps_and_spikes_with_casts.csv")  # 2007-2022 dataset

In [2]:


# =========================================================
# 2. DROP UNUSED COLUMNS
# =========================================================
cols_to_drop_df1 = [
    "Cast_Start",
    "Cast_End",
    "Cast_End_Hour",
    "RadSWD_mean",
    "RadSWD_max",
    "TmpAir_max",
    "TmpAir_mean",
    "HumRel_mean",
    "HumRel_max",
    "PrBaro_mean",
    "PrBaro_max",
    "WndSpd_mean",
    "WndSpd_max",
    "PpRain_mean",
    "PpRain_max",
    "RadClr_mean",
    "RadClr_max",
    "WndDir_max",
    "WndDir_mean",
    "TurbRT_1m"
]

df1 = df1.drop(columns=cols_to_drop_df1, errors="ignore")

In [3]:
# =========================================================
# 3. RENAME df1 COLUMNS TO MATCH df2
# =========================================================
df1 = df1.rename(
    columns={
        "TmpWtr_0p5m": "Water_Temp_0.5",
        "TmpWtr_1m": "Water_Temp_1.0",
        "TmpWtr_2p5m": "Water_Temp_2.5",
        "TmpWtr_4p5m": "Water_Temp_4.5",
        "TmpWtr_6p5m": "Water_Temp_6.5",
        "TmpWtr_8p5m": "Water_Temp_8.5",
        "TmpWtr_10p5m": "Water_Temp_10.5",
        "TmpWtr_12p5m": "Water_Temp_12.5",
        "TmpWtr_14p5m": "Water_Temp_14.5",
        "TmpWtr_16p5m": "Water_Temp_16.5",
        "TmpWtr_18p5m": "Water_Temp_18.5",
        "TmpWtr_20p5m": "Water_Temp_20.5",
        "Cast_Number": "Cast_ID",
    }
)

In [4]:

#making both datasets use same date/time structure so later we can easily merge them together
# =========================================================
# 4. STANDARDIZE DATE / HOUR / IDS
# =========================================================
def standardize_dataset(df, source_name, prefer_existing_datetime=False, dayfirst=False):
    df = df.copy()

    existing_datetime = None
    if prefer_existing_datetime and "datetime" in df.columns:
        existing_datetime = pd.to_datetime(df["datetime"], errors="coerce", dayfirst=dayfirst)

    # Standardize Date
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=dayfirst).dt.normalize()
    else:
        df["Date"] = pd.NaT

    # Numeric fields if present
    if "Cast_Start_Hour" in df.columns:
        df["Cast_Start_Hour"] = pd.to_numeric(df["Cast_Start_Hour"], errors="coerce")

    if "Cast_ID" in df.columns:
        df["Cast_ID"] = pd.to_numeric(df["Cast_ID"], errors="coerce")

    # Infer Cast_Start_Hour if missing
    if "Cast_Start_Hour" not in df.columns and "Cast_ID" in df.columns:
        df["Cast_Start_Hour"] = (df["Cast_ID"] - 1) * 3

    # Infer Cast_ID if missing
    if "Cast_ID" not in df.columns and "Cast_Start_Hour" in df.columns:
        df["Cast_ID"] = (df["Cast_Start_Hour"] // 3) + 1

    # Build datetime
    if existing_datetime is not None:
        df["datetime"] = existing_datetime
    elif "Date" in df.columns and "Cast_Start_Hour" in df.columns:
        df["datetime"] = df["Date"] + pd.to_timedelta(df["Cast_Start_Hour"], unit="h")
    else:
        df["datetime"] = pd.NaT

    df["Date"] = df["datetime"].dt.normalize()
    df["source_dataset"] = source_name #creating this "tracking" column to remember which original dataset each row came from

    return df

df1 = standardize_dataset(df1, source_name="2022_plus", dayfirst=False)
df2 = standardize_dataset(df2, source_name="2007_to_2022", prefer_existing_datetime=True, dayfirst=True)

# =========================================================
# 5. QUICK CHECKS BEFORE MERGING
# =========================================================
print("df1 date range:", df1["datetime"].min(), "to", df1["datetime"].max())
print("df2 date range:", df2["datetime"].min(), "to", df2["datetime"].max())

print("\ndf1 2022 rows:", ((df1["datetime"] >= "2022-01-01") & (df1["datetime"] < "2023-01-01")).sum())
print("df2 2022 rows:", ((df2["datetime"] >= "2022-01-01") & (df2["datetime"] < "2023-01-01")).sum())

df1 date range: 2022-02-21 11:00:00 to 2026-03-12 09:00:00
df2 date range: 2007-07-13 12:00:00 to 2022-01-20 00:00:00

df1 2022 rows: 2166
df2 2022 rows: 153


In [5]:
#This chunk makes sure df1 and df2 have exactly the same columns before combining them.
# =========================================================
# 6. ALIGN COMMON COLUMNS
# =========================================================
common_columns = sorted(set(df1.columns).intersection(df2.columns))

if "datetime" not in common_columns:
    common_columns.append("datetime")

df1 = df1[common_columns].copy()
df2 = df2[common_columns].copy()

In [6]:
print("\nCommon columns:", common_columns)


Common columns: ['Cast_ID', 'Cast_Start_Hour', 'Date', 'S', 'Water_Temp_0.5', 'Water_Temp_1.0', 'Water_Temp_10.5', 'Water_Temp_12.5', 'Water_Temp_14.5', 'Water_Temp_16.5', 'Water_Temp_18.5', 'Water_Temp_2.5', 'Water_Temp_20.5', 'Water_Temp_4.5', 'Water_Temp_6.5', 'Water_Temp_8.5', 'datetime', 'source_dataset']


In [7]:
# =========================================================
# 7. COMBINE DATASETS
# IMPORTANT:
# - use explicit source priority
# - keep 2022+ rows when exact datetimes overlap
# =========================================================
combined = pd.concat([df2, df1], ignore_index=True) #This stacks the two datasets on top of each other.
combined = combined.dropna(subset=["datetime"]).copy() # Remove rows without datetime. 

source_priority = {"2007_to_2022": 0, "2022_plus": 1}
combined["source_priority"] = combined["source_dataset"].map(source_priority).fillna(-1) #If the source name is not found in the dictionary, give it priority -1

In [8]:
duplicates = combined["datetime"].duplicated().sum()
print("Number of duplicate datetimes:", duplicates)
# combined = combined.sort_values(["datetime", "source_priority"]) #For our case, it printed 0 duplicate datetimes, but if your datasets overlap, you should uncomment the following lines 
# combined = combined.drop_duplicates(subset=["datetime"], keep="last").reset_index(drop=True)
# combined = combined.drop(columns=["source_priority"])

Number of duplicate datetimes: 0


In [9]:
#Run these lines to check you haven't lost any rows during the merge
expected_rows = len(df1) + len(df2)
actual_rows = len(combined)

print("df1 rows:", len(df1))
print("df2 rows:", len(df2))
print("Expected combined rows:", expected_rows)
print("Actual combined rows:", actual_rows)
print("Rows lost:", expected_rows - actual_rows)

df1 rows: 10081
df2 rows: 42437
Expected combined rows: 52518
Actual combined rows: 52518
Rows lost: 0


In [ ]:
# =========================================================
# 23. SAVE OUTPUT
# =========================================================
combined = combined.drop(columns=["source_priority", "source_dataset", "Date", "Cast_ID", "Cast_Start_Hour"])
combined.to_excel("fixed_and_profiler_combined_dataset.xlsx", index=False)
print("\nSaved: fixed_and_profiler_combined_dataset.xlsx")


Saved: fixed_and_profiler_combined_dataset.xlsx
